In [111]:
import pandas as pd
import numpy as np
import os
import re
import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# 1. ONLY pull cars that have been successfully deep-scraped!
print("Downloading deep-scraped data from Supabase...")
df = pd.read_sql("SELECT * FROM cars WHERE cylinders IS NOT NULL AND make IS NOT NULL", engine)

# 2. Fill any missing specs with 'unspecified' so OHE doesn't crash
cat_cols = ['condition', 'title_status', 'trim', 'cylinders', 'drive', 'fuel', 'transmission', 'type', 'location']
for col in cat_cols:
    df[col] = df[col].fillna('unspecified')

# 3. Filter outliers
df = df[(df['price'] >= 800) & (df['price'] <= 100000)]
df = df[(df['mileage'] >= 100) & (df['mileage'] <= 300000)]
df = df.dropna(subset=['name', 'price', 'mileage'])

# 4. Extract Year and calculate Age (Still parsing from name for now)
df['year'] = df['name'].astype(str).str.extract(r'(\b(19[0-9]{2}|20[0-2][0-9])\b)')[0]
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)
df['age'] = datetime.datetime.now().year - df['year']

# Clean Location
df['location'] = df['location'].astype(str).str.split('/').str[0].str.lower().str.strip()

print(f"Total clean rows loaded: {len(df)}")
df.head()

Total clean rows loaded: 2261


,name,url,price,mileage,location,make,year,model,age,predicted_price,difference,condition,title_status,trim,region,cylinders,drive,fuel,transmission,type
0,2011 Mercedes-Benz C 350 W 204 Amg Sport Package,https://www.craigslist.org/view/d/yorba-linda-...,7500,122000.0,san fernando valley,mercedes-benz,2011,c-class,15,12211.30,4711.329102,like new,clean,unspecified,other,6 cylinders,rwd,gas,automatic,sedan
1,2018 Audi Q5 Premium Plus,https://www.craigslist.org/view/d/yonkers-2018...,12000,115000.0,yonkers,audi,2018,q5,8,11240.80,-759.191406,unspecified,clean,unspecified,newyork,unspecified,unspecified,gas,automatic,unspecified
2,2017 Ford F-150 Ecoboost Raptor Kit,https://www.craigslist.org/view/d/yonkers-2017...,11500,150000.0,westchester ny,ford,2017,f150,9,13906.60,2406.624023,unspecified,clean,unspecified,other,unspecified,unspecified,gas,automatic,pickup
3,2015 Nissan Rogue Low Miles,https://www.craigslist.org/view/d/yonkers-2015...,6750,110000.0,yonkers,nissan,2015,rogue,11,6567.78,-182.221191,unspecified,clean,unspecified,newyork,unspecified,unspecified,gas,automatic,unspecified
4,2014 Chrysler 300 - Uptown Edition- Low Miles 68K,https://www.craigslist.org/view/d/yonkers-2014...,8600,68000.0,yonkers,chrysler,2014,300,12,9668.64,1068.641602,excellent,clean,unspecified,newyork,6 cylinders,unspecified,gas,automatic,sedan


In [112]:
# Define Features (X) and Target (y)
X = df[['age', 'make', 'model', 'trim', 'mileage', 'location', 'condition', 
        'title_status', 'cylinders', 'drive', 'fuel', 'transmission', 'type']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_features = ['make', 'model', 'trim', 'location', 'condition', 
                'title_status', 'cylinders', 'drive', 'fuel', 'transmission', 'type']

X_train_encoded = ohe.fit_transform(X_train[cat_features])
X_test_encoded = ohe.transform(X_test[cat_features])

X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=ohe.get_feature_names_out(), index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=ohe.get_feature_names_out(), index=X_test.index)

X_train_num = X_train[['age', 'mileage']]
X_test_num = X_test[['age', 'mileage']]

X_train_final = pd.concat([X_train_num, X_train_encoded_df], axis=1)
X_test_final = pd.concat([X_test_num, X_test_encoded_df], axis=1)

print(f"Training matrix shape: {X_train_final.shape}")

Training matrix shape: (1808, 1401)


In [113]:
# Train XGBoost
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

print("Training model on perfect Craigslist data...")
model.fit(X_train_final, y_train)
print("Training complete!")

Training model on perfect Craigslist data...
Training complete!


In [114]:
# Predict
predictions = model.predict(X_test_final)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Model Performance:")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"R-Squared (R2): {r2:.2f}")

# Sample comparison
comparison = pd.DataFrame({
    'Actual_Price': y_test.values[:10],
    'Predicted_Price': predictions[:10].astype(int),
    'Age': X_test['age'].values[:10],
    'Make': X_test['make'].values[:10],
    'Model': X_test['model'].values[:10],
    'Cylinders': X_test['cylinders'].values[:10],
    'Trim': X_test['trim'].values[:10],
    'Drive': X_test['drive'].values[:10],
    'Fuel': X_test['fuel'].values[:10]
})
print("\nSample Predictions vs Actuals:")
print(comparison)

Model Performance:
Mean Absolute Error (MAE): $3,928.80
R-Squared (R2): 0.64

Sample Predictions vs Actuals:
   Actual_Price  Predicted_Price  Age      Make       Model     Cylinders  \
0         11500             9904   19      ford        f550   unspecified   
1         20000            13162   10   porsche       macan   unspecified   
2          6000             5397   21      ford       crown   8 cylinders   
3         24000            14173    7     honda       civic   4 cylinders   
4         10900             9403   14      ford  expedition   8 cylinders   
5         15500            11236   12    toyota        rav4   4 cylinders   
6          6000             9419   13    nissan        juke   unspecified   
7         12500            14123   20      ford       f-450  10 cylinders   
8          2900             2993   17       kia         rio   4 cylinders   
9          1900             1752   22  mercedes        benz   unspecified   

          Trim        Drive Fuel  
0       

In [115]:
# Save artifacts
os.makedirs('../api', exist_ok=True)
joblib.dump(model, '../api/model.pkl')
joblib.dump(ohe, '../api/ohe.pkl')
joblib.dump(X_train_final.columns.tolist(), '../api/model_columns.pkl')
print("\nArtifacts saved successfully!")


Artifacts saved successfully!
